In [ ]:
!pip install -U langchain langchain_openai wikipedia gradio openai

In [ ]:
import os
import wikipedia
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

# Set OpenRouter API key
os.environ["OPENAI_API_KEY"] = "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"

def get_wikipedia_summary(query, sentences=3):
    """Fetches a summary from Wikipedia."""
    try:
        return wikipedia.summary(query, sentences=sentences)
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Multiple results found: {e.options[:5]}..."
    except wikipedia.exceptions.PageError:
        return "No results found."

def initialize_llm():
    """Initializes ChatGPT-3.5 Turbo via OpenRouter API."""
    return ChatOpenAI(
        model="gpt-3.5-turbo",
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0.5
    )

def chatbot_response(query):
    """Fetches a Wikipedia summary and generates a refined response using GPT-3.5 Turbo."""
    llm = initialize_llm()
    summary = get_wikipedia_summary(query)

    system_message = SystemMessage(content="You are an AI assistant providing structured responses based on Wikipedia summaries.")
    user_message = HumanMessage(content=f"Here is a Wikipedia summary: {summary}\nUser Query: {query}\nProvide a helpful response.")

    response = llm.invoke([system_message, user_message])  # Updated method
    return summary, response.content  # Return both Wikipedia summary and AI response

# Gradio UI
with gr.Blocks() as ui:
    gr.Markdown("# 📚 Wikipedia Chatbot 🤖")  # Title

    with gr.Row():
        user_input = gr.Textbox(label="Ask a question about anything:")
        submit_btn = gr.Button("🔍 Search")

    with gr.Row():
        wiki_output = gr.Textbox(label="📖 Wikipedia Summary", interactive=False)

    chat_output = gr.Textbox(label="🤖 Chatbot Response", interactive=False)

    submit_btn.click(chatbot_response, inputs=user_input, outputs=[wiki_output, chat_output])

# Launch the app
ui.launch()
